# Create GNFM Awards (INdAM - Gruppo Nazionale per la Fisica Matematica)

Creates INdAM-GNFM young-researcher project awards ("Progetti Giovani GNFM")
from the group's own approved-project lists, published as PDFs in the GNFM
media library on https://www.altamatematica.it/gnfm/ (attivita/progetti-giovani-gnfm).
A Progetto Giovani is a one-year project led by a young coordinator (RTD,
postdoc or research-contract holder) with 1-4 participants; the money pays the
members' missions and scientific collaborations. **40 projects: 2018 (10),
2019 (12), 2025 (18; 15 + 3 added after extra funding), 100% title /
coordinator / coordinator institution, amount only printed for 2025
(EUR 2,000 each)** (local run 2026-10-01).

Only these three lists are public. The tracker's source URL
(/gnfm/it/archivi/progetti-di-ricerca/) is 404; the "Archivio: Progetti Giovani
GNFM" page names 2006-2011 but links nothing, and neither the pre-2016 Drupal
site nor the Wayback Machine has other lists. The script checks the media
library on every run and fails if a new project-list PDF appears.

GNFM is one of INdAM's national research groups and has its own OpenAlex
funder row; INdAM's own fellowships go under INdAM (F4320311030), GNAMPA and
GNCS have their own ingests (priorities 557, 558). The INdAM "assegni GNFM"
ranking lists (graduatorie degli idonei) are not included: they rank eligible
candidates and do not say which of them were funded.

**Prerequisites:**
- Run `scripts/local/gnfm_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/gnfm/gnfm_projects.parquet` (§1.4 shrink guard).

**Parsing (script):** ruled table rows from pdfplumber, words assigned to the
header columns by x-position (2018, 2019), plain table extract (2025).
Coordinator names are resolved against the GNFM members lists (aderenti, which
print COGNOME and NOME separately); where the PDF has no institution column
(2018, 2025) the coordinator's institution comes from that year's members list
(`lead_institution_source`). 2019 surnames typed with an apostrophe for the
accent (MARO', LIGABO') take the members-list spelling (Marò, Ligabò).

**`funder_award_id` (§2.1.1):** no per-project code is published. Citing works
write "Progetto Giovani GNFM 2023" or the year's shared INdAM CUP code (e.g.
CUP E53C22001930001), so there is nothing project-specific to match. Synthetic
key: `GNFM-{year}-{coordinator-family-slug}`. Collapses onto 0 citation shells.

**Dates:** start_year = end_year = project year; start/end dates NULL.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320334080`
- display_name: Gruppo Nazionale per la Fisica Matematica
- ror_id / doi: from `openalex.funders.funders` (ror 01d6bq278, doi 10.13039/100017143)

**Priority:** `597` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gnfm_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gnfm/gnfm_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.gnfm_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.gnfm_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.gnfm_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320334080 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320334080;

## Step 2: Create GNFM Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gnfm_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320334080  -- GNFM (INdAM)
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        g.*,
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', NULLIF(TRIM(g.lead_country), ''),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_struct
    FROM openalex.awards.gnfm_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('Progetti Giovani GNFM ', g.project_year) as funder_scheme,
    'indam_gnfm_progetti_giovani' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.project_year AS INT) as start_year,
    TRY_CAST(g.project_year AS INT) as end_year,
    g.lead_struct as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    ARRAY(g.lead_struct) as investigators,
    g.source_pdf as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM lead g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'indam_gnfm_progetti_giovani' AND priority = 597;

-- Insert into openalex_awards_raw with priority.
-- Priority 597: direct-from-funder ingest of GNFM's own approved-project lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    597 as priority
FROM openalex.awards.gnfm_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.gnfm_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every key is the synthetic GNFM form.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^GNFM-20[0-9]{2}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as good_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^GNFM-20[0-9]{2}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.gnfm_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.gnfm_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(amount) as with_amount, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.gnfm_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.gnfm_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.gnfm_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/PI/institution, 45% amount: 2025 only).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.gnfm_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'indam_gnfm_progetti_giovani'
GROUP BY provenance, priority;